In [4]:
from pathlib import Path
import os
import torch

print("Python environment restarted successfully")
print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

root = Path("/kaggle/input")

print("\nInput directory exists:", root.exists())

if root.exists():
    print("Attached dataset directories:")
    for p in sorted(root.iterdir()):
        print(" -", p.name)

print("\nChecking expected dataset root:")
expected = root / "datasets" / "saisdsaiiitbh"

print("Exists:", expected.exists())

if expected.exists():
    for p in sorted(expected.iterdir()):
        print(p.name, "| directory:", p.is_dir())

Python environment restarted successfully
PyTorch: 2.11.0+cu128
CUDA available: True

Input directory exists: True
Attached dataset directories:
 - datasets

Checking expected dataset root:
Exists: True
backup-1 | directory: True
dff-code | directory: True
eval-tar | directory: True
ff-tar | directory: True


In [ ]:
from pathlib import Path

BASE = Path("/kaggle/input/datasets/saisdsaiiitbh")

CODE_ROOT = BASE / "dff-code"
FF_ROOT = BASE / "ff-tar"
EVAL_ROOT = BASE / "eval-tar"

for name, root in [
    ("Code", CODE_ROOT),
    ("FF++ crops", FF_ROOT),
    ("Evaluation crops", EVAL_ROOT),
]:
    print(f"\n{'=' * 20} {name} {'=' * 20}")
    print("Path:", root)
    print("Exists:", root.exists())

    if root.exists():
        for item in sorted(root.iterdir())[:20]:
            kind = "DIR " if item.is_dir() else "FILE"
            print(f"  [{kind}] {item.name}")

# Check whether the Python source files are available.
print("\nChecking required source files:")

required = [
    "dff_adapter.py",
    "dataset.py",
    "train.py",
    "evaluate_dff.py",
    "test_dff_adapter.py",
]

for filename in required:
    matches = list(CODE_ROOT.rglob(filename))
    print(f"{filename}: {matches[0] if matches else 'NOT FOUND'}")

In [ ]:
import os

search_roots = [FF_ROOT, EVAL_ROOT]
target_names = {
    "ffpp_manifest_split.csv",
    "cdf1_manifest.csv",
    "cdf2_manifest.csv",
}

found = {}

for root in search_roots:
    if not root.exists():
        continue

    for current, dirs, files in os.walk(root):
        for filename in target_names.intersection(files):
            path = Path(current) / filename
            found.setdefault(filename, path)

        if target_names.issubset(found):
            break

print("Manifest search results:")

for filename in sorted(target_names):
    print(f"{filename}: {found.get(filename, 'NOT FOUND')}")

In [ ]:
import subprocess
import sys

CODE_DIR = next(
    p.parent
    for p in CODE_ROOT.rglob("train.py")
    if all((p.parent / f).exists() for f in [
        "dff_adapter.py",
        "dataset.py",
        "train.py",
        "evaluate_dff.py",
        "test_dff_adapter.py",
    ])
)

print("Using code directory:", CODE_DIR)

result = subprocess.run(
    [sys.executable, "-u", "test_dff_adapter.py"],
    cwd=CODE_DIR,
    text=True,
)

if result.returncode != 0:
    raise RuntimeError("Model tests failed. Do not start training yet.")

print("Model tests passed.")

In [ ]:
import sys
import torch
from pathlib import Path

CODE_DIR = Path("/kaggle/input/datasets/saisdsaiiitbh/dff-code")
FF_ROOT = Path("/kaggle/input/datasets/saisdsaiiitbh/ff-tar")
FF_MANIFEST = FF_ROOT / "ffpp_manifest_split.csv"

EVAL_ROOT = Path("/kaggle/input/datasets/saisdsaiiitbh/eval-tar")
CDF1_MANIFEST = EVAL_ROOT / "cdf1_manifest.csv"
CDF2_MANIFEST = EVAL_ROOT / "cdf2_manifest.csv"

sys.path.insert(0, str(CODE_DIR))

from dataset import FrameDataset

print("GPU:", torch.cuda.get_device_name(0))
print("FF++ manifest exists:", FF_MANIFEST.exists())
print("CDF-v1 manifest exists:", CDF1_MANIFEST.exists())
print("CDF-v2 manifest exists:", CDF2_MANIFEST.exists())

# Construct the dataset index using the existing implementation.
# This may take a little time because it enumerates crop files.
train_ds = FrameDataset(
    crops_root=str(FF_ROOT),
    manifest=str(FF_MANIFEST),
    split="train",
    mode="eval",
    frames_per_video=1,
)

print("\nTraining videos:", len(train_ds.video_labels))
print("Indexed samples:", len(train_ds))

# Read and preprocess one actual crop.
image, authenticity, forgery_type, video_idx = train_ds[0]

print("\nActual crop loaded successfully.")
print("Image shape:", tuple(image.shape))
print("Image dtype:", image.dtype)
print("Authenticity label:", authenticity.item())
print("Forgery-type label:", forgery_type.item())
print("Video index:", video_idx.item())

assert image.shape == (3, 224, 224)
assert torch.isfinite(image).all()
assert authenticity.item() in (0.0, 1.0)
assert 0 <= forgery_type.item() < 5

print("\nReal-data loading and preprocessing passed.")

In [ ]:
import sys
import torch
from torch.utils.data import DataLoader

CODE_DIR = "/kaggle/input/datasets/saisdsaiiitbh/dff-code"
sys.path.insert(0, CODE_DIR)

from dataset import FrameDataset
from train import build_model

device = torch.device("cuda")

# Load two real FF++ training samples.
ds = FrameDataset(
    crops_root=str(FF_ROOT),
    manifest=str(FF_MANIFEST),
    split="train",
    mode="eval",
    frames_per_video=1,
)

loader = DataLoader(ds, batch_size=2, shuffle=False, num_workers=0)
images, auth_labels, type_labels, video_indices = next(iter(loader))

images = images.to(device)

print("Loading DINOv2-Small + DFF-Adapter...")
model = build_model("small", variant="dff").to(device)
model.eval()

print("Model loaded. Running forward pass...")

with torch.inference_mode():
    with torch.autocast(device_type="cuda", dtype=torch.float16):
        auth_logits = model(images, task=0)
        type_logits = model(images, task=1)

auth_logits = auth_logits.float()
type_logits = type_logits.float()

print("\nForward pass successful.")
print("GPU:", torch.cuda.get_device_name(0))
print("Batch shape:", tuple(images.shape))
print("Authenticity logits shape:", tuple(auth_logits.shape))
print("Forgery-type logits shape:", tuple(type_logits.shape))
print("Authenticity logits:", auth_logits.cpu().tolist())
print("Forgery-type predictions:", type_logits.argmax(dim=1).cpu().tolist())

assert auth_logits.shape[0] == 2
assert type_logits.shape == (2, 5)
assert torch.isfinite(auth_logits).all()
assert torch.isfinite(type_logits).all()

del model, images, auth_logits, type_logits, loader, ds
torch.cuda.empty_cache()

print("\nPASS: Real crops + DINOv2-Small + DFF-Adapter work on the T4.")

In [ ]:
import subprocess
import sys

CODE_DIR = "/kaggle/input/datasets/saisdsaiiitbh/dff-code"

for script in ["train.py", "evaluate_dff.py", "run_experiments.py"]:
    print(f"\n{'=' * 25} {script} {'=' * 25}")

    result = subprocess.run(
        [sys.executable, f"{CODE_DIR}/{script}", "--help"],
        capture_output=True,
        text=True,
        timeout=30,
    )

    print(result.stdout[:12000])

    if result.returncode != 0:
        print("STDERR:", result.stderr[:3000])
        print("Return code:", result.returncode)

In [ ]:
import os
import subprocess
import sys
from pathlib import Path

CODE_DIR = "/kaggle/input/datasets/saisdsaiiitbh/dff-code"
FF_ROOT = "/kaggle/working/ff-local"
FF_MANIFEST = f"{FF_ROOT}/ffpp_manifest_split.csv"

RUN_DIR = "/kaggle/working/dff_runs/dff_full"
Path(RUN_DIR).mkdir(parents=True, exist_ok=True)

assert Path(f"{CODE_DIR}/train.py").exists(), "train.py not found"
assert Path(FF_MANIFEST).exists(), "FF++ manifest not found"

cmd = [
    sys.executable, "-u", f"{CODE_DIR}/train.py",
    "--backbone", "small",
    "--crops_root", FF_ROOT,
    "--manifest", FF_MANIFEST,
    "--out_dir", RUN_DIR,
    "--epochs", "12",
    "--frames_per_video", "10",
    "--val_frames", "8",
    "--batch_size", "24",
    "--lr", "1e-4",
    "--weight_decay", "0.01",
    "--lambda_auth", "10",
    "--lambda_type", "2",
    "--label_smoothing", "0.0",
    "--variant", "dff",
    "--num_workers", "4",
    "--max_hours", "6",
    "--ckpt_every", "250",
    "--log_every", "50",
    "--seed", "42",
]

print("Starting main DFF-Adapter training")
print("GPU:", __import__("torch").cuda.get_device_name(0))
print("Output:", RUN_DIR)
print("Maximum training time: 4 hours")
print("-" * 60)

result = subprocess.run(cmd, cwd=CODE_DIR)

print("\nTraining process exit code:", result.returncode)
print("\nFiles saved:")
for p in sorted(Path(RUN_DIR).glob("*")):
    print(f"{p.name:25s} {p.stat().st_size / (1024**2):8.1f} MB")

if result.returncode != 0:
    print("\nTraining exited with an error. Inspect the traceback above before continuing.")

In [ ]:
from pathlib import Path
import json

RUN_DIR = Path("/kaggle/working/dff_runs/dff_full")

print("Checkpoint and result files:")
for p in sorted(RUN_DIR.rglob("*")):
    if p.is_file():
        print(f"{p.relative_to(RUN_DIR)}  ({p.stat().st_size / (1024**2):.1f} MB)")

done_file = RUN_DIR / "done.json"
if done_file.exists():
    print("\nCompletion metadata:")
    print(done_file.read_text())
else:
    print("\nNo done.json found; the run may be incomplete.")

print("\nTraining logs, if saved separately:")
for p in RUN_DIR.glob("*.log"):
    print(p.name)

In [ ]:
import subprocess, sys
CODE_DIR = "/kaggle/input/datasets/saisdsaiiitbh/dff-code"
FF_ROOT = "/kaggle/working/ff-local"
FF_MANIFEST = f"{FF_ROOT}/ffpp_manifest_split.csv"

subprocess.run([sys.executable, "-u", f"{CODE_DIR}/bench.py",
                "--crops_root", FF_ROOT, "--manifest", FF_MANIFEST,
                "--num_workers", "4"], cwd=CODE_DIR)

In [ ]:
import os, shutil, time
from concurrent.futures import ThreadPoolExecutor

SRC = "/kaggle/input/datasets/saisdsaiiitbh/ff-tar"
DST = "/kaggle/working/ff-local"
os.makedirs(DST, exist_ok=True)

files = []
for root, _, fs in os.walk(f"{SRC}/ffpp"):
    for f in fs:
        files.append(os.path.join(root, f))
print("files to copy:", len(files))

def cp(src):
    dst = os.path.join(DST, os.path.relpath(src, SRC))
    if os.path.exists(dst):
        return
    os.makedirs(os.path.dirname(dst), exist_ok=True)
    shutil.copyfile(src, dst)

t = time.time()
with ThreadPoolExecutor(64) as ex:
    list(ex.map(cp, files))
shutil.copyfile(f"{SRC}/ffpp_manifest_split.csv", f"{DST}/ffpp_manifest_split.csv")
print(f"done in {(time.time()-t)/60:.1f} min")
os.system(f"df -h /kaggle/working | tail -1")

In [ ]:
import shutil
from pathlib import Path

source = Path("/kaggle/working/dff_runs/dff_full")
archive_base = "/kaggle/working/dff_full_results"

archive = shutil.make_archive(
    archive_base,
    "zip",
    root_dir=source
)

print("Archive created:", archive)
print("Archive size:", round(Path(archive).stat().st_size / (1024**2), 2), "MB")

In [ ]:
import json
from pathlib import Path

run_dir = Path("/kaggle/working/dff_runs/dff_full")

print("Best checkpoint exists:", (run_dir / "best.pt").exists())
print("Completion metadata:")
print((run_dir / "done.json").read_text())

print("\nTraining configuration:")
print((run_dir / "args.json").read_text())

In [ ]:
import subprocess
import sys
from pathlib import Path

CODE_DIR = "/kaggle/input/datasets/saisdsaiiitbh/dff-code"
EVAL_ROOT = "/kaggle/input/datasets/saisdsaiiitbh/eval-tar"
CKPT = "/kaggle/working/dff_runs/dff_full/best.pt"
OUT = "/kaggle/working/dff_runs/dff_full/cross_dataset_eval.json"

cdf1_manifest = f"{EVAL_ROOT}/cdf1_manifest.csv"
cdf2_manifest = f"{EVAL_ROOT}/cdf2_manifest.csv"

assert Path(CKPT).exists(), f"Checkpoint missing: {CKPT}"
assert Path(cdf1_manifest).exists(), f"Manifest missing: {cdf1_manifest}"
assert Path(cdf2_manifest).exists(), f"Manifest missing: {cdf2_manifest}"

cmd = [
    sys.executable, "-u", f"{CODE_DIR}/evaluate_dff.py",
    "--ckpt", CKPT,
    "--backbone", "small",
    "--variant", "dff",
    "--sets",
    f"{cdf1_manifest}:all",
    f"{cdf2_manifest}:all",
    "--frames", "0",
    "--batch_size", "32",
    "--num_workers", "2",
    "--out", OUT,
]

print("Evaluating best checkpoint on CDF-v1 and CDF-v2...")
result = subprocess.run(cmd, cwd=CODE_DIR)
print("Exit code:", result.returncode)

if result.returncode == 0:
    print("\nEvaluation output:")
    if Path(OUT).exists():
        print(Path(OUT).read_text())
else:
    print("Evaluation failed. Inspect the traceback above before retrying.")

In [ ]:
# Backuop

import os
import shutil

source = "/kaggle/working/dff_runs/dff_full"
backup = "/kaggle/working/dff_baseline_backup.zip"

assert os.path.exists(os.path.join(source, "best.pt")), "Baseline checkpoint missing!"

shutil.make_archive(
    "/kaggle/working/dff_baseline_backup",
    "zip",
    root_dir=source
)

print(f"Backup created: {backup}")
print(f"Size: {os.path.getsize(backup) / (1024**2):.2f} MB")

In [ ]:
# continuing from checkpoint to improve the auc

In [1]:
import os

print(os.path.exists("/kaggle/working/ff-local"))
print(os.path.exists(
    "/kaggle/working/ff-local/ffpp_manifest_split.csv"
))

False
False


In [7]:
import os

for root, dirs, files in os.walk("/kaggle/input"):
    for f in files:
        if f == "last.pt":
            print(os.path.join(root, f))

/kaggle/input/datasets/saisdsaiiitbh/backup-1/last.pt


KeyboardInterrupt: 

In [8]:
import os
import shutil

backup_dir = "/kaggle/input/datasets/saisdsaiiitbh/backup-1"
resume_dir = "/kaggle/working/dff_runs/dff_full_extended"

os.makedirs(resume_dir, exist_ok=True)

# Locate last.pt, even if it is inside a subdirectory
source_checkpoint = None

for root, dirs, files in os.walk(backup_dir):
    if "last.pt" in files:
        source_checkpoint = os.path.join(root, "last.pt")
        break

if source_checkpoint is None:
    raise FileNotFoundError(
        f"last.pt not found anywhere inside {backup_dir}"
    )

destination = os.path.join(resume_dir, "last.pt")
shutil.copy2(source_checkpoint, destination)

print("Source:", source_checkpoint)
print("Checkpoint restored:", os.path.exists(destination))
print("Size:", round(os.path.getsize(destination) / (1024**2), 2), "MB")

Source: /kaggle/input/datasets/saisdsaiiitbh/backup-1/last.pt
Checkpoint restored: True
Size: 7.8 MB


In [9]:
import os

print("Crops available:",
      os.path.isdir("/kaggle/working/ff-local"))

print("Manifest available:",
      os.path.isfile(
          "/kaggle/working/ff-local/ffpp_manifest_split.csv"
      ))


Crops available: False
Manifest available: False


In [11]:
import os

for root, dirs, files in os.walk("/kaggle/input"):
    depth = root.replace("/kaggle/input", "").count(os.sep)

    if depth > 2:
        dirs[:] = []
        continue

    print(f"\n📁 {root}")

    for f in files[:20]:
        print(f"   {f}")


📁 /kaggle/input

📁 /kaggle/input/datasets

📁 /kaggle/input/datasets/saisdsaiiitbh


In [13]:
from pathlib import Path

ff_source = Path("/kaggle/input/datasets/saisdsaiiitbh/ff-tar")

assert ff_source.is_dir(), f"Dataset not found: {ff_source}"

print("FF++ source:", ff_source)

for p in sorted(ff_source.rglob("*")):
    if p.is_file():
        print(f"{p.stat().st_size / (1024**3):8.3f} GB  {p.relative_to(ff_source)}")

FF++ source: /kaggle/input/datasets/saisdsaiiitbh/ff-tar


KeyboardInterrupt: 

In [14]:
from pathlib import Path

ff_source = Path("/kaggle/input/datasets/saisdsaiiitbh/ff-tar")

print("Source exists:", ff_source.is_dir())
print("\nTop-level contents:")

for p in sorted(ff_source.iterdir()):
    if p.is_dir():
        children = list(p.iterdir())
        print(f"[DIR]  {p.name}/ ({len(children)} immediate items)")
        for child in children[:5]:
            print("       ", child.name)
    else:
        print(f"[FILE] {p.name} ({p.stat().st_size / (1024**2):.2f} MB)")

Source exists: True

Top-level contents:
[DIR]  ffpp/ (5 immediate items)
        Deepfakes
        Face2Face
        FaceSwap
        NeuralTextures
        original
[FILE] ffpp_manifest_split.csv (0.28 MB)


In [1]:
import os
import time
from concurrent.futures import ThreadPoolExecutor
from tqdm.auto import tqdm

SRC = "/kaggle/input/datasets/saisdsaiiitbh/ff-tar"
DST = "/kaggle/working/ff-local"

SRC_CROPS = os.path.join(SRC, "ffpp")
DST_CROPS = os.path.join(DST, "ffpp")
SRC_MANIFEST = os.path.join(SRC, "ffpp_manifest_split.csv")
DST_MANIFEST = os.path.join(DST, "ffpp_manifest_split.csv")

os.makedirs(DST_CROPS, exist_ok=True)

files = []
for root, _, filenames in os.walk(SRC_CROPS):
    for filename in filenames:
        src = os.path.join(root, filename)
        dst = os.path.join(DST_CROPS, os.path.relpath(src, SRC_CROPS))
        files.append((src, dst))

print(f"Files found: {len(files):,}")

def copy_file(item):
    src, dst = item

    if os.path.isfile(dst) and os.path.getsize(dst) == os.path.getsize(src):
        return "skipped"

    os.makedirs(os.path.dirname(dst), exist_ok=True)
    temp_dst = dst + ".copying"

    try:
        with open(src, "rb") as fsrc, open(temp_dst, "wb") as fdst:
            while True:
                chunk = fsrc.read(1024 * 1024)
                if not chunk:
                    break
                fdst.write(chunk)

        os.replace(temp_dst, dst)
        return "copied"

    finally:
        if os.path.exists(temp_dst):
            os.remove(temp_dst)

start = time.time()
copied = 0
skipped = 0
errors = 0

with ThreadPoolExecutor(max_workers=32) as executor:
    results = executor.map(copy_file, files)

    for result in tqdm(results, total=len(files), desc="Copying FF++ crops"):
        if result == "copied":
            copied += 1
        else:
            skipped += 1

print(f"\nCopied: {copied:,}")
print(f"Skipped (already present): {skipped:,}")
print(f"Elapsed: {(time.time() - start) / 60:.1f} minutes")

# Copy manifest
if not (
    os.path.isfile(DST_MANIFEST)
    and os.path.getsize(DST_MANIFEST) == os.path.getsize(SRC_MANIFEST)
):
    import shutil
    shutil.copy2(SRC_MANIFEST, DST_MANIFEST)

print("Manifest ready:", os.path.isfile(DST_MANIFEST))
os.system("df -h /kaggle/working | tail -1")

Files found: 165,000


Copying FF++ crops:   0%|          | 0/165000 [00:00<?, ?it/s]


Copied: 165,000
Skipped (already present): 0
Elapsed: 1.1 minutes
Manifest ready: True
/dev/loop2       20G  2.5G   18G  13% /kaggle/working


0

In [2]:
from pathlib import Path
import csv
import torch

# Paths
data_dir = Path("/kaggle/working/ff-local")
crops_dir = data_dir / "ffpp"
manifest = data_dir / "ffpp_manifest_split.csv"
checkpoint = Path("/kaggle/working/dff_runs/dff_full_extended/last.pt")
code_dir = Path("/kaggle/input/datasets/saisdsaiiitbh/dff-code")
train_script = code_dir / "train.py"

# 1. Verify dataset
print("=== DATASET ===")
print("Crops directory exists:", crops_dir.is_dir())
print("Manifest exists:", manifest.is_file())

if crops_dir.is_dir():
    print("Crop files:", sum(1 for p in crops_dir.rglob("*") if p.is_file()))

if manifest.is_file():
    with manifest.open("r", newline="") as f:
        reader = csv.reader(f)
        header = next(reader, None)
        rows = sum(1 for _ in reader)
    print("Manifest columns:", header)
    print("Manifest rows:", rows)

# 2. Verify training code
print("\n=== TRAINING CODE ===")
print("Code directory exists:", code_dir.is_dir())
print("train.py exists:", train_script.is_file())

if code_dir.is_dir():
    print("Code files:", [p.name for p in code_dir.iterdir()])

# 3. Inspect checkpoint
print("\n=== CHECKPOINT ===")
print("Checkpoint exists:", checkpoint.is_file())

if checkpoint.is_file():
    print(f"Checkpoint size: {checkpoint.stat().st_size / (1024**2):.1f} MiB")
    ckpt = torch.load(checkpoint, map_location="cpu", weights_only=False)
    print("Checkpoint keys:", list(ckpt.keys()))

    for key in ("epoch", "step", "global_step", "best_val_auc", "config"):
        if key in ckpt:
            value = ckpt[key]
            if key == "config":
                print("config:", value)
            else:
                print(f"{key}: {value}")

=== DATASET ===
Crops directory exists: True
Manifest exists: True
Crop files: 165000
Manifest columns: ['dataset', 'method', 'video_id', 'label', 'forgery_type', 'split', 'n_frames', 'path']
Manifest rows: 5000

=== TRAINING CODE ===
Code directory exists: True
train.py exists: True
Code files: ['train.py', 'run_experiments.py', 'test_dff_adapter.py', 'dff_adapter.py', 'bench.py', 'evaluate_dff.py', 'dataset.py']

=== CHECKPOINT ===
Checkpoint exists: False


In [18]:
import torch
from pathlib import Path

checkpoint_path = Path(
    "/kaggle/working/dff_runs/dff_full_extended/last.pt"
)

ckpt = torch.load(
    checkpoint_path,
    map_location="cpu",
    weights_only=False
)

print("Saved epoch:", ckpt["epoch"])
print("Global step:", ckpt["step"])
print("Best validation AUC:", ckpt["best_auc"])

print("\nOriginal training arguments:")
for key, value in ckpt["args"].items():
    print(f"{key}: {value}")

Saved epoch: 12
Global step: 18000
Best validation AUC: 0.990829081632653

Original training arguments:
backbone: small
crops_root: /kaggle/working/ff-local
manifest: /kaggle/working/ff-local/ffpp_manifest_split.csv
out_dir: /kaggle/working/dff_runs/dff_full
epochs: 12
frames_per_video: 10
val_frames: 8
batch_size: 24
lr: 0.0001
weight_decay: 0.01
lambda_auth: 10.0
lambda_type: 2.0
label_smoothing: 0.0
no_aux: False
variant: dff
num_workers: 4
max_hours: 6.0
ckpt_every: 250
log_every: 50
seed: 42
smoke: False


In [19]:
from pathlib import Path

train_file = Path(
    "/kaggle/input/datasets/saisdsaiiitbh/dff-code/train.py"
)

lines = train_file.read_text().splitlines()

keywords = [
    "ArgumentParser",
    "add_argument",
    "resume",
    "torch.load",
    "load_state_dict",
    "optimizer",
    "scheduler",
    "start_epoch",
    "range(",
    "args.epochs",
    "max_hours",
    "last.pt",
    "best.pt",
    "torch.save",
    "out_dir",
]

for i, line in enumerate(lines):
    if any(keyword in line for keyword in keywords):
        start = max(0, i - 2)
        end = min(len(lines), i + 4)
        print(f"\n--- train.py lines {start + 1}-{end} ---")
        for j in range(start, end):
            print(f"{j + 1:4}: {lines[j]}")


--- train.py lines 10-15 ---
  10: Built to survive Kaggle/Colab session limits:
  11:   * checkpoint every --ckpt_every steps and at every epoch end (atomic write: temp file then rename)
  12:   * on startup, resumes from <out_dir>/last.pt if it exists (also mid-epoch)
  13:   * --max_hours: saves and exits cleanly before the session limit  (prints TIME_LIMIT_REACHED)
  14:   * checkpoints hold only the trainable weights (a few MB), so handing over to a teammate = copying last.pt
  15: 

--- train.py lines 11-16 ---
  11:   * checkpoint every --ckpt_every steps and at every epoch end (atomic write: temp file then rename)
  12:   * on startup, resumes from <out_dir>/last.pt if it exists (also mid-epoch)
  13:   * --max_hours: saves and exits cleanly before the session limit  (prints TIME_LIMIT_REACHED)
  14:   * checkpoints hold only the trainable weights (a few MB), so handing over to a teammate = copying last.pt
  15: 
  16: Examples

--- train.py lines 12-17 ---
  12:   * on startu

In [20]:
from pathlib import Path
import torch
import shutil

code_dir = Path("/kaggle/input/datasets/saisdsaiiitbh/dff-code")
source_ckpt = Path("/kaggle/working/dff_runs/dff_full_extended/last.pt")
test_dir = Path("/kaggle/working/dff_runs/dff_resume_test")

assert (code_dir / "train.py").exists(), "train.py not found"
assert source_ckpt.exists(), "Resume checkpoint not found"

ck = torch.load(source_ckpt, map_location="cpu", weights_only=False)

print("Checkpoint epoch:", ck["epoch"])
print("Global step:", ck["step"])
print("Best validation AUC:", ck["best_auc"])
print("Saved backbone:", ck["args"]["backbone"])
print("Saved frames/video:", ck["args"]["frames_per_video"])
print("Saved batch size:", ck["args"]["batch_size"])

test_dir.mkdir(parents=True, exist_ok=True)
shutil.copy2(source_ckpt, test_dir / "last.pt")

print("\nTest checkpoint ready:", test_dir / "last.pt")

Checkpoint epoch: 12
Global step: 18000
Best validation AUC: 0.990829081632653
Saved backbone: small
Saved frames/video: 10
Saved batch size: 24

Test checkpoint ready: /kaggle/working/dff_runs/dff_resume_test/last.pt


In [21]:
%cd /kaggle/input/datasets/saisdsaiiitbh/dff-code

!python -u train.py \
    --backbone small \
    --crops_root /kaggle/working/ff-local \
    --manifest /kaggle/working/ff-local/ffpp_manifest_split.csv \
    --out_dir /kaggle/working/dff_runs/dff_resume_test \
    --epochs 13 \
    --frames_per_video 10 \
    --val_frames 8 \
    --batch_size 24 \
    --lr 0.0001 \
    --weight_decay 0.01 \
    --lambda_auth 10.0 \
    --lambda_type 2.0 \
    --label_smoothing 0.0 \
    --num_workers 4 \
    --max_hours 2 \
    --ckpt_every 250 \
    --log_every 50 \
    --seed 42

/kaggle/input/datasets/saisdsaiiitbh/dff-code
train: 3600 videos -> 36000 samples/epoch | val: 700 videos, 5600 frames | device=cuda amp=True
config.json: 100%|█████████████████████████████| 547/547 [00:00<00:00, 2.29MB/s]

model.safetensors: downloading bytes: ███████████████████▋  | 78.7MB, 6.87MB/s  
model.safetensors: downloading bytes: ██████████████████████| 83.7MB, 7.79MB/s  
model.safetensors: reconstructing file: 100%|██████| 88.2MB / 88.2MB, 8.52MB/s  
Loading weights: 100%|██████████████████████| 223/223 [00:00<00:00, 2789.65it/s]
trainable parameters: 667,374
RESUMED from /kaggle/working/dff_runs/dff_resume_test/last.pt: epoch=12 batch=0 step=18000 best_auc=0.9908
plan: 13 epochs x 1500 steps = 19500 steps (batch 24, 36000 samples/epoch)
epoch 12: first batch ready after 0.2s
{'epoch': 12, 'step': 18050, 'of': 19500, 'loss_auth': 0.0852, 'loss_type': 0.1225, 'acc_type': 0.953, 'samples_s': 89.9, 'eta_min': 6.5}
{'epoch': 12, 'step': 18100, 'of': 19500, 'loss_auth': 0.1006, 

In [22]:
from pathlib import Path
import shutil

source = Path("/kaggle/working/dff_runs/dff_resume_test/last.pt")
target_dir = Path("/kaggle/working/dff_runs/dff_full_extended")
target = target_dir / "last.pt"

assert source.exists(), "Resume-test checkpoint is missing"
target_dir.mkdir(parents=True, exist_ok=True)

shutil.copy2(source, target)

print("Checkpoint copied:", target)

Checkpoint copied: /kaggle/working/dff_runs/dff_full_extended/last.pt


In [23]:
%cd /kaggle/input/datasets/saisdsaiiitbh/dff-code

!python -u train.py \
    --backbone small \
    --crops_root /kaggle/working/ff-local \
    --manifest /kaggle/working/ff-local/ffpp_manifest_split.csv \
    --out_dir /kaggle/working/dff_runs/dff_full_extended \
    --epochs 30 \
    --frames_per_video 10 \
    --val_frames 8 \
    --batch_size 24 \
    --lr 0.0001 \
    --weight_decay 0.01 \
    --lambda_auth 10.0 \
    --lambda_type 2.0 \
    --label_smoothing 0.0 \
    --num_workers 4 \
    --max_hours 5 \
    --ckpt_every 250 \
    --log_every 50 \
    --seed 42

/kaggle/input/datasets/saisdsaiiitbh/dff-code
train: 3600 videos -> 36000 samples/epoch | val: 700 videos, 5600 frames | device=cuda amp=True
Loading weights: 100%|██████████████████████| 223/223 [00:00<00:00, 2669.78it/s]
trainable parameters: 667,374
RESUMED from /kaggle/working/dff_runs/dff_full_extended/last.pt: epoch=13 batch=0 step=19500 best_auc=0.9908
plan: 30 epochs x 1500 steps = 45000 steps (batch 24, 36000 samples/epoch)
epoch 13: first batch ready after 0.2s
{'epoch': 13, 'step': 19550, 'of': 45000, 'loss_auth': 0.0724, 'loss_type': 0.1203, 'acc_type': 0.958, 'samples_s': 98.2, 'eta_min': 103.7}
{'epoch': 13, 'step': 19600, 'of': 45000, 'loss_auth': 0.0984, 'loss_type': 0.1406, 'acc_type': 0.958, 'samples_s': 104.2, 'eta_min': 97.5}
{'epoch': 13, 'step': 19650, 'of': 45000, 'loss_auth': 0.08, 'loss_type': 0.1233, 'acc_type': 0.953, 'samples_s': 103.0, 'eta_min': 98.4}
{'epoch': 13, 'step': 19700, 'of': 45000, 'loss_auth': 0.0832, 'loss_type': 0.1432, 'acc_type': 0.949, 'sa

In [28]:
import json
from pathlib import Path

run_dir = Path("/kaggle/working/dff_runs/dff_resume_test")

print("Best checkpoint exists:", (run_dir / "best.pt").exists())
print("Completion metadata:")
print((run_dir / "done.json").read_text())

print("\nTraining configuration:")
print((run_dir / "args.json").read_text())

Best checkpoint exists: False
Completion metadata:
{"best_val_auc": 0.990829081632653, "steps": 19500}

Training configuration:
{
  "backbone": "small",
  "crops_root": "/kaggle/working/ff-local",
  "manifest": "/kaggle/working/ff-local/ffpp_manifest_split.csv",
  "out_dir": "/kaggle/working/dff_runs/dff_resume_test",
  "epochs": 13,
  "frames_per_video": 10,
  "val_frames": 8,
  "batch_size": 24,
  "lr": 0.0001,
  "weight_decay": 0.01,
  "lambda_auth": 10.0,
  "lambda_type": 2.0,
  "label_smoothing": 0.0,
  "no_aux": false,
  "variant": "dff",
  "num_workers": 4,
  "max_hours": 2.0,
  "ckpt_every": 250,
  "log_every": 50,
  "seed": 42,
  "smoke": false
}


In [29]:
from pathlib import Path
import torch

paths = [
    Path("/kaggle/input/datasets/saisdsaiiitbh/backup-1/best.pt"),
    Path("/kaggle/input/datasets/saisdsaiiitbh/backup-1/last.pt"),
    Path("/kaggle/working/dff_runs/dff_resume_test/last.pt"),
    Path("/kaggle/working/dff_runs/dff_full_extended/last.pt"),
]

for path in paths:
    print(f"\n{path}")
    print("Exists:", path.exists())

    if path.exists():
        print(f"Size: {path.stat().st_size / (1024**2):.2f} MiB")

        checkpoint = torch.load(
            path, map_location="cpu", weights_only=False
        )

        print("Epoch:", checkpoint.get("epoch"))
        print("Step:", checkpoint.get("step"))
        print("Best AUC:", checkpoint.get("best_auc"))


/kaggle/input/datasets/saisdsaiiitbh/backup-1/best.pt
Exists: True
Size: 7.80 MiB
Epoch: 11
Step: 16500
Best AUC: 0.990829081632653

/kaggle/input/datasets/saisdsaiiitbh/backup-1/last.pt
Exists: True
Size: 7.80 MiB
Epoch: 12
Step: 18000
Best AUC: 0.990829081632653

/kaggle/working/dff_runs/dff_resume_test/last.pt
Exists: True
Size: 7.80 MiB
Epoch: 13
Step: 19500
Best AUC: 0.990829081632653

/kaggle/working/dff_runs/dff_full_extended/last.pt
Exists: True
Size: 7.80 MiB
Epoch: 30
Step: 45000
Best AUC: 0.990829081632653


## checking cross validation

In [30]:
from pathlib import Path
import shutil

source = Path(
    "/kaggle/input/datasets/saisdsaiiitbh/backup-1/best.pt"
)

destination = Path(
    "/kaggle/working/dff_runs/dff_full_extended/best.pt"
)

assert source.exists(), f"Source checkpoint not found: {source}"

shutil.copy2(source, destination)

print("Restored:", destination)
print(f"Size: {destination.stat().st_size / (1024**2):.2f} MiB")
print("Checkpoint exists:", destination.exists())

Restored: /kaggle/working/dff_runs/dff_full_extended/best.pt
Size: 7.80 MiB
Checkpoint exists: True


In [31]:
import subprocess
import sys
from pathlib import Path

CODE_DIR = "/kaggle/input/datasets/saisdsaiiitbh/dff-code"
EVAL_ROOT = "/kaggle/input/datasets/saisdsaiiitbh/eval-tar"
RUN_ROOT = Path("/kaggle/working/dff_runs")

cdf1_manifest = f"{EVAL_ROOT}/cdf1_manifest.csv"
cdf2_manifest = f"{EVAL_ROOT}/cdf2_manifest.csv"

# Original best model and extended final model
checkpoints = {
    "original_best": RUN_ROOT / "dff_full_extended" / "best.pt",
    "extended_last": RUN_ROOT / "dff_full_extended" / "last.pt",
}

for manifest in [cdf1_manifest, cdf2_manifest]:
    assert Path(manifest).exists(), f"Manifest missing: {manifest}"

for name, ckpt in checkpoints.items():
    assert ckpt.exists(), f"Checkpoint missing: {ckpt}"

    out = RUN_ROOT / "dff_full_extended" / f"{name}_cross_dataset_eval.json"

    cmd = [
        sys.executable,
        "-u",
        f"{CODE_DIR}/evaluate_dff.py",
        "--ckpt", str(ckpt),
        "--backbone", "small",
        "--variant", "dff",
        "--sets",
        f"{cdf1_manifest}:all",
        f"{cdf2_manifest}:all",
        "--frames", "0",
        "--batch_size", "32",
        "--num_workers", "2",
        "--out", str(out),
    ]

    print("\n" + "=" * 70)
    print(f"Evaluating: {name}")
    print(f"Checkpoint: {ckpt}")
    print("=" * 70)

    result = subprocess.run(cmd, cwd=CODE_DIR)

    print("Exit code:", result.returncode)

    if result.returncode == 0 and out.exists():
        print(f"\nResults for {name}:")
        print(out.read_text())
    else:
        print(f"Evaluation failed for {name}. Inspect the traceback above.")
        break


Evaluating: original_best
Checkpoint: /kaggle/working/dff_runs/dff_full_extended/best.pt


Loading weights: 100%|██████████| 223/223 [00:00<00:00, 2697.12it/s]


/kaggle/input/datasets/saisdsaiiitbh/eval-tar/cdf1_manifest.csv:all AUC=81.20  EER=30.65  frameAUC=77.47  (100 videos, 3200 frames)
/kaggle/input/datasets/saisdsaiiitbh/eval-tar/cdf2_manifest.csv:all AUC=85.02  EER=22.94  frameAUC=77.71  (518 videos, 16576 frames)
Exit code: 0

Results for original_best:
{
  "/kaggle/input/datasets/saisdsaiiitbh/eval-tar/cdf1_manifest.csv:all": {
    "video_auc": 0.8119694397283531,
    "video_eer": 0.30645161290322576,
    "frame_auc": 0.774682244137309,
    "n_videos": 100,
    "n_frames": 3200
  },
  "/kaggle/input/datasets/saisdsaiiitbh/eval-tar/cdf2_manifest.csv:all": {
    "video_auc": 0.8502313284864508,
    "video_eer": 0.22941176470588232,
    "frame_auc": 0.7770855466813656,
    "n_videos": 518,
    "n_frames": 16576
  }
}

Evaluating: extended_last
Checkpoint: /kaggle/working/dff_runs/dff_full_extended/last.pt


Loading weights: 100%|██████████| 223/223 [00:00<00:00, 2701.29it/s]


/kaggle/input/datasets/saisdsaiiitbh/eval-tar/cdf1_manifest.csv:all AUC=82.98  EER=26.32  frameAUC=79.56  (100 videos, 3200 frames)
/kaggle/input/datasets/saisdsaiiitbh/eval-tar/cdf2_manifest.csv:all AUC=86.25  EER=22.35  frameAUC=79.58  (518 videos, 16576 frames)
Exit code: 0

Results for extended_last:
{
  "/kaggle/input/datasets/saisdsaiiitbh/eval-tar/cdf1_manifest.csv:all": {
    "video_auc": 0.8297962648556876,
    "video_eer": 0.2631578947368421,
    "frame_auc": 0.795561863327674,
    "n_videos": 100,
    "n_frames": 3200
  },
  "/kaggle/input/datasets/saisdsaiiitbh/eval-tar/cdf2_manifest.csv:all": {
    "video_auc": 0.8624586913417053,
    "video_eer": 0.22352941176470587,
    "frame_auc": 0.7958377896124215,
    "n_videos": 518,
    "n_frames": 16576
  }
}


In [32]:
# Backuop 2

import os
import shutil
import zipfile
from pathlib import Path

source = Path("/kaggle/working/dff_runs")
backup_base = "/kaggle/working/dff_baseline_backup2"

assert source.is_dir(), f"Run directory not found: {source}"

# List the files that will be backed up.
print("Files to back up:")
for path in sorted(source.rglob("*")):
    if path.is_file():
        print(f"  {path.relative_to(source)}")

# Create the ZIP.
backup = shutil.make_archive(
    backup_base,
    "zip",
    root_dir=str(source),
)

# Verify archive integrity.
with zipfile.ZipFile(backup, "r") as archive:
    assert archive.testzip() is None, "ZIP integrity check failed!"

print(f"\nBackup created: {backup}")
print(f"Size: {os.path.getsize(backup) / (1024**2):.2f} MB")
print("ZIP integrity check passed.")

Files to back up:
  dff_full_extended/args.json
  dff_full_extended/best.pt
  dff_full_extended/done.json
  dff_full_extended/extended_last_cross_dataset_eval.json
  dff_full_extended/last.pt
  dff_full_extended/log.jsonl
  dff_full_extended/original_best_cross_dataset_eval.json
  dff_resume_test/args.json
  dff_resume_test/done.json
  dff_resume_test/last.pt
  dff_resume_test/log.jsonl

Backup created: /kaggle/working/dff_baseline_backup2.zip
Size: 19.76 MB
ZIP integrity check passed.


In [33]:
from pathlib import Path

paths = [
    Path("/kaggle/input/datasets/saisdsaiiitbh/backup-1/best.pt"),
    Path("/kaggle/input/datasets/saisdsaiiitbh/backup-1/last.pt"),
    Path("/kaggle/working/dff_runs/dff_full_extended/best.pt"),
    Path("/kaggle/working/dff_runs/dff_full_extended/last.pt"),
]

for path in paths:
    print(f"{path}: {'EXISTS' if path.is_file() else 'MISSING'}")

/kaggle/input/datasets/saisdsaiiitbh/backup-1/best.pt: EXISTS
/kaggle/input/datasets/saisdsaiiitbh/backup-1/last.pt: EXISTS
/kaggle/working/dff_runs/dff_full_extended/best.pt: EXISTS
/kaggle/working/dff_runs/dff_full_extended/last.pt: EXISTS


# trying dinov2base

In [34]:
from pathlib import Path

code_dir = Path("/kaggle/input/datasets/saisdsaiiitbh/dff-code")

for filename in ["run_experiments.py", "train.py", "dff_adapter.py"]:
    path = code_dir / filename
    print(f"{filename}: {'EXISTS' if path.exists() else 'MISSING'}")

if (code_dir / "run_experiments.py").exists():
    text = (code_dir / "run_experiments.py").read_text()
    print("\n--- Experiment definitions ---")

    for i, line in enumerate(text.splitlines(), 1):
        if (
            "COMMON =" in line
            or "EXPERIMENTS =" in line
            or '"dff_base"' in line
            or "def train_cmd" in line
            or "def eval_cmd" in line
        ):
            print(f"{i}: {line}")

run_experiments.py: EXISTS
train.py: EXISTS
dff_adapter.py: EXISTS

--- Experiment definitions ---
25: COMMON = ["--backbone", "small", "--crops_root", FF_ROOT, "--epochs", "12", "--frames_per_video", "16",
29: EXPERIMENTS = {
37:     "dff_base":    ["--backbone", "base", "--epochs", "6"],   # stretch: DINOv2-Base (speed unmeasured)
45: def train_cmd(name):
54: def eval_cmd(name):


In [36]:
!python - <<'PY'
from transformers import AutoModel

model = AutoModel.from_pretrained("facebook/dinov2-base")

print("Model:", model.config._name_or_path)
print("Hidden size:", model.config.hidden_size)
print("Transformer blocks:", len(model.encoder.layer))
print("Attention heads:", model.config.num_attention_heads)

for name, param in model.named_parameters():
    print("First parameter:", name, tuple(param.shape))
    break
PY

/bin/bash: line 1: warning: here-document at line 1 delimited by end-of-file (wanted `PY')


config.json:   0%|          | 0.00/548 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  346MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/223 [00:00<?, ?it/s]

Model: facebook/dinov2-base
Hidden size: 768
Transformer blocks: 12
Attention heads: 12
First parameter: embeddings.cls_token (1, 1, 768)


NameError: name 'PY' is not defined

In [81]:
import os

os.chdir("/")
print("Current directory:", os.getcwd())

Current directory: /


In [82]:
os.chdir("/kaggle/input/datasets/saisdsaiiitbh/dff-code")
print("Current directory:", os.getcwd())

Current directory: /kaggle/input/datasets/saisdsaiiitbh/dff-code


In [83]:
import os

print("Current directory:", os.getcwd())
print("Root accessible:", os.path.exists("/"))
print("Working directory accessible:", os.path.isdir("/kaggle/working"))
print("Input directory accessible:", os.path.isdir("/kaggle/input"))

Current directory: /kaggle/input/datasets/saisdsaiiitbh/dff-code
Root accessible: True
Working directory accessible: True
Input directory accessible: True


In [75]:
!python test_dff_adapter.py

PASS  test_adapters_wrap_q_v_dense_in_every_block
PASS  test_each_input_chunk_used_exactly_once
PASS  test_gradient_flow_and_unselected_logits
PASS  test_linear_probe_variant
PASS  test_lora_baseline_variant
PASS  test_matches_naive_equations
PASS  test_only_adapters_and_heads_train
      Small  adapter params =    665,064
      Base   adapter params =  1,328,616
      Large  adapter params =  3,541,968
PASS  test_parameter_budget_matches_paper
      never-selected (adapter,head,expert) slots: 28/144 = 19.4%
PASS  test_routing_report_runs
PASS  test_task_switch_changes_output
PASS  test_top3_routing_and_normalisation
PASS  test_zero_init_is_identity

12/12 passed


In [76]:
!python test_dff_adapter.py --real

PASS  test_adapters_wrap_q_v_dense_in_every_block
PASS  test_each_input_chunk_used_exactly_once
PASS  test_gradient_flow_and_unselected_logits
PASS  test_linear_probe_variant
PASS  test_lora_baseline_variant
PASS  test_matches_naive_equations
PASS  test_only_adapters_and_heads_train
      Small  adapter params =    665,064
      Base   adapter params =  1,328,616
      Large  adapter params =  3,541,968
PASS  test_parameter_budget_matches_paper
Loading weights: 100%|██████████████████████| 223/223 [00:00<00:00, 2748.54it/s]
      trainable=1,333,230 of total=87,913,710 (1.52%)
      GPU peak memory, batch 16, fp16, both passes: 2.88 GiB
PASS  test_real_dinov2_small
      never-selected (adapter,head,expert) slots: 22/144 = 15.3%
PASS  test_routing_report_runs
PASS  test_task_switch_changes_output
PASS  test_top3_routing_and_normalisation
PASS  test_zero_init_is_identity

13/13 passed


In [77]:
!python run_experiments.py dff_base --dry_run

[GPU 0] dff_base: /usr/bin/python3 train.py --crops_root /kaggle/input/datasets/saisdsaiiitbh/ff-crops --frames_per_video 16 --val_frames 8 --batch_size 24 --num_workers 4 --ckpt_every 500 --log_every 50 --max_hours 5 --out_dir runs/dff_base --backbone base --epochs 12
[GPU 0] eval dff_base: /usr/bin/python3 evaluate_dff.py --ckpt runs/dff_base/best.pt --backbone base --variant dff --sets /kaggle/input/datasets/saisdsaiiitbh/ff-crops/ffpp_manifest_split.csv:test /kaggle/input/datasets/saisdsaiiitbh/eval-corps/cdf1_manifest.csv:all /kaggle/input/datasets/saisdsaiiitbh/eval-corps/cdf2_manifest.csv:all --num_workers 4 --out runs/dff_base/results.json


In [84]:
!grep -n -A8 -B3 '"dff_base"' run_experiments.py

69-        "--epochs", "12",
70-        "--variant", "linear",
71-    ],
72:    "dff_base": [
73-        "--backbone", "base",
74-        "--epochs", "12",
75-    ],
76-}
77-
78-EVAL_SETS = [f"{FF_ROOT}/ffpp_manifest_split.csv:test",
79-             f"{EV_ROOT}/cdf1_manifest.csv:all",
80-             f"{EV_ROOT}/cdf2_manifest.csv:all"]


In [85]:
!grep -n -A15 '^COMMON' run_experiments.py

25:COMMON = [
26-    "--crops_root", FF_ROOT,
27-    "--frames_per_video", "16",
28-    "--val_frames", "8",
29-    "--batch_size", "24",
30-    "--num_workers", "4",
31-    "--ckpt_every", "500",
32-    "--log_every", "50",
33-    "--max_hours", "8",
34-]
35-
36-EXPERIMENTS = {
37-    "dff_full": [
38-        "--backbone", "small",
39-        "--epochs", "12",
40-    ],


In [86]:
!ls -lah runs/dff_base 2>/dev/null

In [87]:
from pathlib import Path

print("Dataset:", Path("/kaggle/working/ff-local/ffpp").is_dir())
print("Manifest:", Path("/kaggle/working/ff-local/ffpp_manifest_split.csv").is_file())
print("Output directory exists:",
      Path("/kaggle/working/dff_runs/dff_small_current").exists())

Dataset: True
Manifest: True
Output directory exists: False


In [5]:
from pathlib import Path
import shutil

src = Path("/kaggle/input/datasets/saisdsaiiitbh/dff-code")
dst = Path("/kaggle/working/dff-code")

if not src.exists():
    raise FileNotFoundError(f"Source directory not found: {src}")

if dst.exists():
    print(f"Using existing working copy: {dst}")
else:
    shutil.copytree(src, dst)
    print(f"Copied source code to: {dst}")

print("\nFiles:")
for p in sorted(dst.iterdir()):
    print(p.name)

Copied source code to: /kaggle/working/dff-code

Files:
bench.py
dataset.py
dff_adapter.py
evaluate_dff.py
run_experiments.py
test_dff_adapter.py
train.py


In [6]:
import torch

print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
print("Number of GPUs:", torch.cuda.device_count())

for i in range(torch.cuda.device_count()):
    print(f"GPU {i}: {torch.cuda.get_device_name(i)}")

PyTorch: 2.11.0+cu128
CUDA available: True
Number of GPUs: 2
GPU 0: Tesla T4
GPU 1: Tesla T4


In [7]:
from pathlib import Path

train_file = Path("/kaggle/working/dff-code/train.py")
source = train_file.read_text()

for i, line in enumerate(source.splitlines(), start=1):
    if (
        "model = build_model" in line
        or "params = model.trainable_parameters()" in line
        or "model(x, task=0)" in line
        or "model(x, task=1)" in line
    ):
        print(f"{i}: {line}")

146:     model = build_model(args.backbone, variant=args.variant).to(device)
147:     params = model.trainable_parameters()
192:                 logit = model(x, task=0)                                           # pass 1: authenticity
198:                     type_logits = model(x, task=1).float()                         # pass 2: forgery type


In [8]:
from pathlib import Path

train_file = Path("/kaggle/working/dff-code/train.py")
source = train_file.read_text()

# Make sure we're patching the expected version of train.py.
assert source.count("model = build_model(args.backbone, variant=args.variant).to(device)") == 1
assert source.count("logit = model(x, task=0)") == 1
assert source.count("type_logits = model(x, task=1).float()") == 1

# Keep the original model object for optimizer setup, checkpoint loading,
# saving, and validation. Use a wrapper only for training forward passes.
old_init = "model = build_model(args.backbone, variant=args.variant).to(device)"
new_init = '''model = build_model(args.backbone, variant=args.variant).to(device)
    n_gpus = torch.cuda.device_count() if device.type == "cuda" else 0
    if n_gpus > 1:
        train_model = torch.nn.DataParallel(
            model,
            device_ids=list(range(n_gpus)),
            output_device=0,
        )
        print(f"Multi-GPU training enabled: {n_gpus} GPUs")
    else:
        train_model = model
        print(f"Single-GPU training: {device}")'''

source = source.replace(old_init, new_init, 1)
source = source.replace("logit = model(x, task=0)", "logit = train_model(x, task=0)", 1)
source = source.replace(
    "type_logits = model(x, task=1).float()",
    "type_logits = train_model(x, task=1).float()",
    1,
)

train_file.write_text(source)
print("Patched:", train_file)

Patched: /kaggle/working/dff-code/train.py


In [9]:
from pathlib import Path

source = Path("/kaggle/working/dff-code/train.py").read_text()

checks = {
    "DataParallel configured": "torch.nn.DataParallel(" in source,
    "Authenticity pass uses wrapper": "logit = train_model(x, task=0)" in source,
    "Forgery-type pass uses wrapper": "type_logits = train_model(x, task=1).float()" in source,
    "Original trainable-parameter selection preserved": "params = model.trainable_parameters()" in source,
}

for name, passed in checks.items():
    print(f"{'PASS' if passed else 'FAIL'}: {name}")

assert all(checks.values()), "A code check failed; do not start training."

PASS: DataParallel configured
PASS: Authenticity pass uses wrapper
PASS: Forgery-type pass uses wrapper
PASS: Original trainable-parameter selection preserved


In [10]:
import py_compile

py_compile.compile(
    "/kaggle/working/dff-code/train.py",
    doraise=True,
)

print("Python syntax check passed.")

Python syntax check passed.


In [11]:
%cd /kaggle/working/dff-code

!python -u train.py \
    --backbone base \
    --crops_root /kaggle/working/ff-local \
    --manifest /kaggle/working/ff-local/ffpp_manifest_split.csv \
    --out_dir /kaggle/working/dff_runs/dff_base_current \
    --epochs 30 \
    --frames_per_video 8 \
    --val_frames 8 \
    --batch_size 24 \
    --lr 0.0001 \
    --weight_decay 0.01 \
    --lambda_auth 10.0 \
    --lambda_type 2.0 \
    --label_smoothing 0.0 \
    --num_workers 8 \
    --max_hours 8 \
    --ckpt_every 500 \
    --log_every 200 \
    --seed 42

/kaggle/working/dff-code
train: 3600 videos -> 28800 samples/epoch | val: 700 videos, 5600 frames | device=cuda amp=True
Loading weights: 100%|██████████████████████| 223/223 [00:00<00:00, 2923.60it/s]
Multi-GPU training enabled: 2 GPUs
trainable parameters: 1,333,230
RESUMED from /kaggle/working/dff_runs/dff_base_current/last.pt: epoch=1 batch=300 step=1500 best_auc=0.9730
plan: 30 epochs x 1200 steps = 36000 steps (batch 24, 28800 samples/epoch)
/usr/local/lib/python3.13/dist-packages/torch/utils/data/dataloader.py:424: UserWarning: This DataLoader will create 8 worker processes in total. Our suggested max number of worker in current system is 4, which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running slow or even freeze, lower the worker number to avoid potential slowness/freeze if necessary.
  self.check_worker_number_rationality()
/usr/local/lib/python3.13/dist-packages/torch/utils/data/dataloader.p

In [18]:
import subprocess
import sys
from pathlib import Path

# ============================================================
# 1. Paths
# ============================================================

BASE = Path("/kaggle/input/datasets/saisdsaiiitbh")

# Use the modified code copied into the writable working directory.
CODE_DIR = Path("/kaggle/working/dff-code")

# Base model checkpoint directory.
RUN_DIR = Path("/kaggle/working/dff_runs/dff_base_current")

CKPT = RUN_DIR / "best.pt"
OUT = RUN_DIR / "cross_dataset_eval.json"

# ============================================================
# 2. Locate evaluation manifests
# ============================================================

eval_candidates = [
    BASE / "eval-tar",
    BASE / "eval-corps",
]

EVAL_ROOT = next(
    (
        p for p in eval_candidates
        if (p / "cdf1_manifest.csv").is_file()
        and (p / "cdf2_manifest.csv").is_file()
    ),
    None,
)

if EVAL_ROOT is None:
    raise FileNotFoundError(
        "Could not find both cdf1_manifest.csv and cdf2_manifest.csv "
        "inside either:\n"
        f"  {BASE / 'eval-tar'}\n"
        f"  {BASE / 'eval-corps'}"
    )

CDF1_MANIFEST = EVAL_ROOT / "cdf1_manifest.csv"
CDF2_MANIFEST = EVAL_ROOT / "cdf2_manifest.csv"

# ============================================================
# 3. Validate inputs
# ============================================================

EVALUATOR = CODE_DIR / "evaluate_dff.py"

required_files = {
    "Evaluation script": EVALUATOR,
    "Best checkpoint": CKPT,
    "Celeb-DF-v1 manifest": CDF1_MANIFEST,
    "Celeb-DF-v2 manifest": CDF2_MANIFEST,
}

print("=" * 70)
print("DFF-Adapter Cross-Dataset Evaluation")
print("=" * 70)

for name, path in required_files.items():
    print(f"{name}: {path}")
    if not path.is_file():
        raise FileNotFoundError(f"{name} not found: {path}")

# Ensure the output directory exists.
RUN_DIR.mkdir(parents=True, exist_ok=True)

print("\nConfiguration")
print("-" * 70)
print("Backbone: DINOv2-Base")
print("Variant: DFF-Adapter")
print("Checkpoint:", CKPT)
print("Evaluation root:", EVAL_ROOT)
print("Output JSON:", OUT)
print("Batch size: 16")
print("DataLoader workers: 2")
print("Frames: all frames, if supported by evaluate_dff.py")
print("=" * 70)

# ============================================================
# 4. Build evaluation command
# ============================================================

cmd = [
    sys.executable,
    "-u",
    str(EVALUATOR),
    "--ckpt", str(CKPT),
    "--backbone", "base",
    "--variant", "dff",
    "--sets",
    f"{CDF1_MANIFEST}:all",
    f"{CDF2_MANIFEST}:all",
    "--frames", "0",
    "--batch_size", "16",
    "--num_workers", "2",
    "--out", str(OUT),
]

print("\nCommand:")
print(" ".join(cmd))
print("\nStarting evaluation...\n")

# ============================================================
# 5. Run evaluation
# ============================================================

result = subprocess.run(
    cmd,
    cwd=str(CODE_DIR),
    check=False,
)

print("\n" + "=" * 70)
print("Evaluation process finished")
print("=" * 70)
print("Exit code:", result.returncode)

# ============================================================
# 6. Display and validate results
# ============================================================

if result.returncode != 0:
    raise RuntimeError(
        "Cross-dataset evaluation failed. "
        "Inspect the traceback above before retrying."
    )

if not OUT.is_file():
    raise FileNotFoundError(
        f"Evaluation returned exit code 0, but the expected output "
        f"JSON was not created: {OUT}"
    )

print("\nCross-dataset evaluation results:")
print("-" * 70)
print(OUT.read_text())

print("\nResults saved to:")
print(OUT)

DFF-Adapter Cross-Dataset Evaluation
Evaluation script: /kaggle/working/dff-code/evaluate_dff.py
Best checkpoint: /kaggle/working/dff_runs/dff_base_current/best.pt
Celeb-DF-v1 manifest: /kaggle/input/datasets/saisdsaiiitbh/eval-tar/cdf1_manifest.csv
Celeb-DF-v2 manifest: /kaggle/input/datasets/saisdsaiiitbh/eval-tar/cdf2_manifest.csv

Configuration
----------------------------------------------------------------------
Backbone: DINOv2-Base
Variant: DFF-Adapter
Checkpoint: /kaggle/working/dff_runs/dff_base_current/best.pt
Evaluation root: /kaggle/input/datasets/saisdsaiiitbh/eval-tar
Output JSON: /kaggle/working/dff_runs/dff_base_current/cross_dataset_eval.json
Batch size: 16
DataLoader workers: 2
Frames: all frames, if supported by evaluate_dff.py

Command:
/usr/bin/python3 -u /kaggle/working/dff-code/evaluate_dff.py --ckpt /kaggle/working/dff_runs/dff_base_current/best.pt --backbone base --variant dff --sets /kaggle/input/datasets/saisdsaiiitbh/eval-tar/cdf1_manifest.csv:all /kaggle/i

Loading weights: 100%|██████████| 223/223 [00:00<00:00, 2798.30it/s]


/kaggle/input/datasets/saisdsaiiitbh/eval-tar/cdf1_manifest.csv:all AUC=81.71  EER=28.95  frameAUC=75.44  (100 videos, 3200 frames)
/kaggle/input/datasets/saisdsaiiitbh/eval-tar/cdf2_manifest.csv:all AUC=84.22  EER=23.82  frameAUC=77.66  (518 videos, 16576 frames)

Evaluation process finished
Exit code: 0

Cross-dataset evaluation results:
----------------------------------------------------------------------
{
  "/kaggle/input/datasets/saisdsaiiitbh/eval-tar/cdf1_manifest.csv:all": {
    "video_auc": 0.817062818336163,
    "video_eer": 0.2894736842105263,
    "frame_auc": 0.7544210592635824,
    "n_videos": 100,
    "n_frames": 3200
  },
  "/kaggle/input/datasets/saisdsaiiitbh/eval-tar/cdf2_manifest.csv:all": {
    "video_auc": 0.84216787838731,
    "video_eer": 0.2382352941176471,
    "frame_auc": 0.7765639119170316,
    "n_videos": 518,
    "n_frames": 16576
  }
}

Results saved to:
/kaggle/working/dff_runs/dff_base_current/cross_dataset_eval.json


In [19]:
# Checking paths
import json
from pathlib import Path

paths = [
    Path("/kaggle/working/dff_runs/dff_base_current/args.json"),
    Path("/kaggle/working/dff_runs/dff_full_extended/args.json"),
]

for path in paths:
    print(f"\n{'=' * 60}")
    print(path)

    if path.exists():
        print(json.dumps(json.loads(path.read_text()), indent=2))
    else:
        print("args.json not found")


/kaggle/working/dff_runs/dff_base_current/args.json
{
  "backbone": "base",
  "crops_root": "/kaggle/working/ff-local",
  "manifest": "/kaggle/working/ff-local/ffpp_manifest_split.csv",
  "out_dir": "/kaggle/working/dff_runs/dff_base_current",
  "epochs": 30,
  "frames_per_video": 8,
  "val_frames": 8,
  "batch_size": 24,
  "lr": 0.0001,
  "weight_decay": 0.01,
  "lambda_auth": 10.0,
  "lambda_type": 2.0,
  "label_smoothing": 0.0,
  "no_aux": false,
  "variant": "dff",
  "num_workers": 8,
  "max_hours": 8.0,
  "ckpt_every": 500,
  "log_every": 200,
  "seed": 42,
  "smoke": false
}

/kaggle/working/dff_runs/dff_full_extended/args.json
args.json not found


In [22]:
# BAckups
import json
import zipfile
from pathlib import Path
from IPython.display import display, FileLink

# Main paths
WORK_DIR = Path("/kaggle/working")
RUN_DIR = WORK_DIR / "dff_runs" / "dff_base_current"
CODE_DIR = Path("/kaggle/input/datasets/saisdsaiiitbh/dff-code")

BACKUP_PATH = WORK_DIR / "dff_base_30epoch_backup_2.zip"

# Verify the experiment directory exists.
if not RUN_DIR.is_dir():
    raise FileNotFoundError(f"Training output directory missing: {RUN_DIR}")

print("Base experiment files found:")
for path in sorted(RUN_DIR.rglob("*")):
    if path.is_file():
        print(f"  {path.relative_to(RUN_DIR)} "
              f"({path.stat().st_size / 1024**2:.2f} MB)")

# Warn if the expected checkpoints are missing.
for name in ["best.pt", "last.pt"]:
    path = RUN_DIR / name
    if not path.is_file():
        print(f"WARNING: {name} is missing.")

# Package experiment outputs and relevant source scripts.
with zipfile.ZipFile(
    BACKUP_PATH, "w", compression=zipfile.ZIP_DEFLATED
) as archive:

    # Training checkpoints, logs, arguments, completion metadata,
    # and cross-dataset evaluation JSON, if present.
    for path in sorted(RUN_DIR.rglob("*")):
        if path.is_file():
            archive.write(
                path,
                arcname=str(Path("dff_base_current") / path.relative_to(RUN_DIR))
            )

    # Include the scripts used for reproducibility.
    for name in ["train.py", "evaluate_dff.py"]:
        path = CODE_DIR / name
        if path.is_file():
            archive.write(path, arcname=f"source/{name}")

# Validate archive integrity.
with zipfile.ZipFile(BACKUP_PATH, "r") as archive:
    bad_file = archive.testzip()
    if bad_file is not None:
        raise RuntimeError(f"ZIP integrity check failed: {bad_file}")

    print("\nFiles inside backup:")
    for name in archive.namelist():
        print(" ", name)

print(f"\nBackup created successfully: {BACKUP_PATH}")
print(f"ZIP size: {BACKUP_PATH.stat().st_size / 1024**2:.2f} MB")

# Display a clickable download link in the notebook.
display(FileLink(str(BACKUP_PATH), result_html_prefix="Download your Base experiment backup: "))

Base experiment files found:
  args.json (0.00 MB)
  best.pt (15.42 MB)
  cross_dataset_eval.json (0.00 MB)
  done.json (0.00 MB)
  last.pt (15.42 MB)
  log.jsonl (0.03 MB)

Files inside backup:
  dff_base_current/args.json
  dff_base_current/best.pt
  dff_base_current/cross_dataset_eval.json
  dff_base_current/done.json
  dff_base_current/last.pt
  dff_base_current/log.jsonl
  source/train.py
  source/evaluate_dff.py

Backup created successfully: /kaggle/working/dff_base_30epoch_backup_2.zip
ZIP size: 27.07 MB


/kaggle/working/dff_base_30epoch_backup_2.zip

In [21]:
# prevent session from stopping
import os
import time
from datetime import datetime

LOG_FILE = "/kaggle/working/session_keepalive.log"
INTERVAL = 15

print("=" * 60)
print("Training and packaging should now be complete.")
print("Starting heartbeat logger.")
print(f"Log file: {LOG_FILE}")
print("Interrupt this cell when you want to stop.")
print("=" * 60)

while True:
    timestamp = datetime.now().strftime("%Y-%m-%d %H:%M:%S")

    with open(LOG_FILE, "a") as f:
        f.write(f"{timestamp} | Session heartbeat\n")
        f.flush()
        os.fsync(f.fileno())

    print(f"[{timestamp}] Heartbeat written")
    time.sleep(INTERVAL)

Training and packaging should now be complete.
Starting heartbeat logger.
Log file: /kaggle/working/session_keepalive.log
Interrupt this cell when you want to stop.
[2026-10-29 07:04:57] Heartbeat written
[2026-10-29 07:05:13] Heartbeat written
[2026-10-29 07:05:28] Heartbeat written
[2026-10-29 07:05:43] Heartbeat written
[2026-10-29 07:05:58] Heartbeat written
[2026-10-29 07:06:13] Heartbeat written
[2026-10-29 07:06:28] Heartbeat written
[2026-10-29 07:06:43] Heartbeat written
[2026-10-29 07:06:58] Heartbeat written
[2026-10-29 07:07:13] Heartbeat written
[2026-10-29 07:07:28] Heartbeat written
[2026-10-29 07:07:43] Heartbeat written
[2026-10-29 07:07:58] Heartbeat written
[2026-10-29 07:08:13] Heartbeat written
[2026-10-29 07:08:28] Heartbeat written
[2026-10-29 07:08:43] Heartbeat written
[2026-10-29 07:08:58] Heartbeat written
[2026-10-29 07:09:13] Heartbeat written
[2026-10-29 07:09:28] Heartbeat written
[2026-10-29 07:09:43] Heartbeat written
[2026-10-29 07:09:58] Heartbeat wri

KeyboardInterrupt: 

In [23]:
from pathlib import Path

run_dir = Path("/kaggle/working/dff_runs/dff_base_current")

for filename in ["args.json", "best.pt", "last.pt", "cross_dataset_eval.json"]:
    path = run_dir / filename
    print(f"{filename}: {'OK' if path.is_file() else 'MISSING'}")

args.json: OK
best.pt: OK
last.pt: OK
cross_dataset_eval.json: OK


In [25]:
from pathlib import Path
import torch
import json

# ------------------------------------------------------------
# 1. Locate the current checkpoint
# ------------------------------------------------------------

CURRENT_DIR = Path("/kaggle/working/dff_runs/dff_base_current")
CURRENT_BEST = CURRENT_DIR / "best.pt"
CURRENT_LAST = CURRENT_DIR / "last.pt"

print("=" * 75)
print("CURRENT BASE CHECKPOINTS")
print("=" * 75)

for path in [CURRENT_BEST, CURRENT_LAST]:
    if path.is_file():
        print(f"FOUND: {path} ({path.stat().st_size / 1024**2:.2f} MB)")
    else:
        print(f"MISSING: {path}")

# ------------------------------------------------------------
# 2. Search attached Kaggle datasets for backup checkpoints
# ------------------------------------------------------------

print("\n" + "=" * 75)
print("SEARCHING KAGGLE INPUT FOR BACKUPS")
print("=" * 75)

backup_candidates = []

input_root = Path("/kaggle/input")

if input_root.exists():
    for path in input_root.rglob("*.pt"):
        # Avoid listing duplicate references.
        if path.is_file():
            backup_candidates.append(path)

if not backup_candidates:
    print("No .pt checkpoint files found under /kaggle/input.")
else:
    for path in sorted(set(backup_candidates)):
        size_mb = path.stat().st_size / 1024**2
        print(f"{size_mb:8.2f} MB  {path}")

# ------------------------------------------------------------
# 3. Inspect checkpoint structure
# ------------------------------------------------------------

def load_checkpoint(path):
    """
    Load a trusted checkpoint onto CPU for inspection.
    weights_only=False is used because training checkpoints can
    contain metadata in addition to tensors.
    """
    try:
        checkpoint = torch.load(
            path,
            map_location="cpu",
            weights_only=False,
        )
    except TypeError:
        # Compatibility with older PyTorch versions.
        checkpoint = torch.load(path, map_location="cpu")

    print(f"\nReadable checkpoint: {path}")
    print("Top-level type:", type(checkpoint).__name__)

    if isinstance(checkpoint, dict):
        print("Top-level keys:", list(checkpoint.keys())[:40])

        for key in [
            "epoch",
            "step",
            "global_step",
            "best_auc",
            "best_val_auc",
        ]:
            if key in checkpoint:
                print(f"{key}: {checkpoint[key]}")

    return checkpoint


print("\n" + "=" * 75)
print("TEST CURRENT BEST CHECKPOINT")
print("=" * 75)

if CURRENT_BEST.is_file():
    current_checkpoint = load_checkpoint(CURRENT_BEST)
else:
    current_checkpoint = None
    print("Cannot inspect current best checkpoint: file is missing.")

# ------------------------------------------------------------
# 4. Inspect likely Base-model backups
# ------------------------------------------------------------

print("\n" + "=" * 75)
print("INSPECTING LIKELY BASE BACKUPS")
print("=" * 75)

likely_backups = [
    path for path in sorted(set(backup_candidates))
    if (
        "dff_base_current" in str(path).lower()
        or (
            "base" in str(path).lower()
            and path.name in {"best.pt", "last.pt"}
        )
    )
]

if not likely_backups:
    print("No clearly identifiable Base-model backup found.")
    print("A checkpoint may still exist under a differently named directory.")
else:
    for backup_path in likely_backups:
        try:
            backup_checkpoint = load_checkpoint(backup_path)
            print("BACKUP LOAD TEST: PASSED")
        except Exception as exc:
            print("BACKUP LOAD TEST: FAILED")
            print(type(exc).__name__ + ":", exc)

# ------------------------------------------------------------
# 5. Display metadata and evaluation results
# ------------------------------------------------------------

print("\n" + "=" * 75)
print("RUN METADATA")
print("=" * 75)

for filename in ["args.json", "cross_dataset_eval.json"]:
    path = CURRENT_DIR / filename
    if path.is_file():
        print(f"\n--- {filename} ---")
        print(path.read_text())
    else:
        print(f"MISSING: {path}")

print("\nDiagnostic complete.")
print("No checkpoints were modified.")

CURRENT BASE CHECKPOINTS
FOUND: /kaggle/working/dff_runs/dff_base_current/best.pt (15.42 MB)
FOUND: /kaggle/working/dff_runs/dff_base_current/last.pt (15.42 MB)

SEARCHING KAGGLE INPUT FOR BACKUPS
    7.80 MB  /kaggle/input/datasets/saisdsaiiitbh/backup-1/best.pt
    7.80 MB  /kaggle/input/datasets/saisdsaiiitbh/backup-1/last.pt

TEST CURRENT BEST CHECKPOINT

Readable checkpoint: /kaggle/working/dff_runs/dff_base_current/best.pt
Top-level type: dict
Top-level keys: ['model', 'opt', 'scaler', 'epoch', 'batches_done', 'step', 'best_auc', 'args']
epoch: 26
step: 31200
best_auc: 0.991594387755102

INSPECTING LIKELY BASE BACKUPS
No clearly identifiable Base-model backup found.
A checkpoint may still exist under a differently named directory.

RUN METADATA

--- args.json ---
{
  "backbone": "base",
  "crops_root": "/kaggle/working/ff-local",
  "manifest": "/kaggle/working/ff-local/ffpp_manifest_split.csv",
  "out_dir": "/kaggle/working/dff_runs/dff_base_current",
  "epochs": 30,
  "frames_pe

In [27]:
from pathlib import Path
import hashlib
import torch

CURRENT = Path("/kaggle/working/dff_runs/dff_base_current/best.pt")
BACKUP_DIR = Path("/kaggle/input/datasets/saisdsaiiitbh/backup-1")
BACKUP_BEST = BACKUP_DIR / "best.pt"
BACKUP_LAST = BACKUP_DIR / "last.pt"

def load_ckpt(path):
    try:
        return torch.load(path, map_location="cpu", weights_only=False)
    except TypeError:
        return torch.load(path, map_location="cpu")

def sha256(path):
    h = hashlib.sha256()
    with path.open("rb") as f:
        for chunk in iter(lambda: f.read(1024 * 1024), b""):
            h.update(chunk)
    return h.hexdigest()

def inspect(path):
    print(f"\n{'=' * 70}")
    print(path)
    print(f"{'=' * 70}")

    if not path.is_file():
        print("FILE MISSING")
        return None

    print(f"File size: {path.stat().st_size / 1024**2:.2f} MB")
    print("SHA-256:", sha256(path))

    try:
        ckpt = load_ckpt(path)
    except Exception as e:
        print("LOAD FAILED:", repr(e))
        return None

    print("Load: SUCCESS")
    print("Top-level type:", type(ckpt).__name__)

    if isinstance(ckpt, dict):
        print("Top-level keys:", list(ckpt.keys()))

        args = ckpt.get("args")
        if args is not None:
            print("\nRecorded training arguments:")
            if isinstance(args, dict):
                for k, v in args.items():
                    print(f"  {k}: {v}")
            elif hasattr(args, "__dict__"):
                for k, v in vars(args).items():
                    print(f"  {k}: {v}")
            else:
                print(args)

        for k in ["epoch", "step", "batches_done", "best_auc"]:
            if k in ckpt:
                print(f"{k}: {ckpt[k]}")

        model = ckpt.get("model")
        if isinstance(model, dict):
            tensors = {
                k: v for k, v in model.items()
                if isinstance(v, torch.Tensor)
            }
            print("\nModel state entries:", len(model))
            print("Tensor entries:", len(tensors))
            print("First 12 model keys:")
            for k in list(model.keys())[:12]:
                v = model[k]
                print(f"  {k}: {tuple(v.shape) if isinstance(v, torch.Tensor) else type(v).__name__}")

    return ckpt

current = inspect(CURRENT)
backup_best = inspect(BACKUP_BEST)
backup_last = inspect(BACKUP_LAST)

# Compare the model-state dictionaries.
def get_model_state(ckpt):
    if not isinstance(ckpt, dict):
        return None
    state = ckpt.get("model")
    return state if isinstance(state, dict) else None

def compare(label, a, b):
    print(f"\n--- Comparison: current best vs {label} ---")
    sa, sb = get_model_state(a), get_model_state(b)

    if sa is None or sb is None:
        print("Cannot compare: one checkpoint has no 'model' state dictionary.")
        return

    ka, kb = set(sa), set(sb)
    print("Current model entries:", len(ka))
    print("Other model entries:", len(kb))
    print("Same key set:", ka == kb)

    if ka != kb:
        print("Keys only in current:", list(ka - kb)[:10])
        print("Keys only in other:", list(kb - ka)[:10])
        print("Architecture compatibility: NOT ESTABLISHED")
        return

    mismatches = [
        k for k in sa
        if isinstance(sa[k], torch.Tensor)
        and isinstance(sb[k], torch.Tensor)
        and sa[k].shape != sb[k].shape
    ]

    non_tensors = [
        k for k in sa
        if not isinstance(sa[k], torch.Tensor)
        or not isinstance(sb[k], torch.Tensor)
    ]

    print("Tensor shape mismatches:", len(mismatches))
    if mismatches:
        for k in mismatches[:10]:
            print(k, sa[k].shape, sb[k].shape)

    print("Non-tensor entries requiring review:", len(non_tensors))

    if not mismatches and not non_tensors:
        same_values = all(
            torch.equal(sa[k], sb[k])
            for k in sa
        )
        print("All model tensors exactly equal:", same_values)
        print(
            "Architecture compatibility: model keys and tensor shapes match"
            if not mismatches
            else "Architecture compatibility: not established"
        )

if current is not None and backup_best is not None:
    compare("backup best.pt", current, backup_best)

if current is not None and backup_last is not None:
    compare("backup last.pt", current, backup_last)


/kaggle/working/dff_runs/dff_base_current/best.pt
File size: 15.42 MB
SHA-256: e80e76fd8750c9f6ce7356618ffaf5fcfc68d5c0ec995eb10eba5d07e50dfc0b
Load: SUCCESS
Top-level type: dict
Top-level keys: ['model', 'opt', 'scaler', 'epoch', 'batches_done', 'step', 'best_auc', 'args']

Recorded training arguments:
  backbone: base
  crops_root: /kaggle/working/ff-local
  manifest: /kaggle/working/ff-local/ffpp_manifest_split.csv
  out_dir: /kaggle/working/dff_runs/dff_base_current
  epochs: 30
  frames_per_video: 8
  val_frames: 8
  batch_size: 24
  lr: 0.0001
  weight_decay: 0.01
  lambda_auth: 10.0
  lambda_type: 2.0
  label_smoothing: 0.0
  no_aux: False
  variant: dff
  num_workers: 8
  max_hours: 8.0
  ckpt_every: 500
  log_every: 200
  seed: 42
  smoke: False
epoch: 26
step: 31200
batches_done: 0
best_auc: 0.991594387755102

Model state entries: 148
Tensor entries: 148
First 12 model keys:
  backbone.encoder.layer.0.attention.attention.query.adapter.A: (4, 6, 4, 192)
  backbone.encoder.lay

In [28]:
from pathlib import Path
import zipfile

RUN_DIR = Path("/kaggle/working/dff_runs/dff_base_current")
ARCHIVE = Path("/kaggle/working/dff_base_current_backup.zip")

files_to_save = [
    "best.pt",
    "last.pt",
    "args.json",
    "cross_dataset_eval.json",
]

missing = [
    name for name in files_to_save
    if not (RUN_DIR / name).is_file()
]

if missing:
    raise FileNotFoundError(f"Missing required files: {missing}")

with zipfile.ZipFile(
    ARCHIVE, "w", compression=zipfile.ZIP_STORED
) as zf:
    for name in files_to_save:
        path = RUN_DIR / name
        zf.write(path, arcname=f"dff_base_current/{name}")

print("Backup archive created:", ARCHIVE)
print(f"Archive size: {ARCHIVE.stat().st_size / 1024**2:.2f} MB")

with zipfile.ZipFile(ARCHIVE, "r") as zf:
    print("Archive integrity check:", zf.testzip() or "PASSED")
    print("Contents:")
    for name in zf.namelist():
        print(" -", name)

Backup archive created: /kaggle/working/dff_base_current_backup.zip
Archive size: 30.85 MB
Archive integrity check: PASSED
Contents:
 - dff_base_current/best.pt
 - dff_base_current/last.pt
 - dff_base_current/args.json
 - dff_base_current/cross_dataset_eval.json


# Checking dfdc

In [1]:
from pathlib import Path

print("=== Kaggle input datasets ===")
for p in sorted(Path("/kaggle/input").iterdir()):
    print(p)

print("\n=== Kaggle working directory ===")
for p in sorted(Path("/kaggle/working").iterdir()):
    print(p)

print("\n=== Available GPUs ===")
!nvidia-smi

=== Kaggle input datasets ===
/kaggle/input/datasets

=== Kaggle working directory ===
/kaggle/working/.virtual_documents

=== Available GPUs ===
Thu Oct 29 17:40:42 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.178.04             Driver Version: 580.178.04     CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   50C    P8             10W /   70W |       0MiB /  15360MiB |      0%      Defa

In [4]:
# Locate dfdc and generate its manifest

from pathlib import Path
import csv
import os
import time

INPUT_ROOT = Path("/kaggle/input/datasets/saisdsaiiitbh")
OUTPUT_MANIFEST = Path("/kaggle/working/dfdc_manifest.csv")

# Your Kaggle dataset should normally be mounted here.
expected_root = INPUT_ROOT / "dfdc-crops" / "dfdc"

if expected_root.is_dir():
    dfdc_dir = expected_root
else:
    # Search only a few directory levels instead of the entire dataset.
    candidates = []
    for dataset_dir in INPUT_ROOT.iterdir():
        if not dataset_dir.is_dir():
            continue
        candidate = dataset_dir / "dfdc"
        if (candidate / "REAL").is_dir() and (candidate / "FAKE").is_dir():
            candidates.append(candidate)

    if len(candidates) != 1:
        print("Could not uniquely locate the DFDC dataset.")
        print("Available Kaggle inputs:")
        for p in INPUT_ROOT.iterdir():
            print(" ", p)
        raise RuntimeError(
            "Set dfdc_dir manually to the directory containing REAL and FAKE."
        )

    dfdc_dir = candidates[0]

assert (dfdc_dir / "REAL").is_dir(), "REAL directory not found"
assert (dfdc_dir / "FAKE").is_dir(), "FAKE directory not found"

print("DFDC directory:", dfdc_dir, flush=True)

rows = []
skipped_videos = []
start = time.time()

for class_name, label in [("REAL", 0), ("FAKE", 1)]:
    class_dir = dfdc_dir / class_name
    video_dirs = sorted(p for p in class_dir.iterdir() if p.is_dir())

    print(
        f"\nScanning {class_name}: {len(video_dirs)} video folders",
        flush=True
    )

    for i, video_dir in enumerate(video_dirs, start=1):
        # Match dataset.py: it reads files ending in lowercase '.jpg'.
        n_frames = sum(
            1 for name in os.listdir(video_dir)
            if name.endswith(".jpg")
            and (video_dir / name).is_file()
        )

        if n_frames == 0:
            skipped_videos.append(str(video_dir))
            continue

        rows.append({
            "dataset": "dfdc",
            "method": "dfdc",
            "video_id": video_dir.name,
            "label": label,
            "forgery_type": 1 if label == 1 else 0,
            "split": "test",
            "n_frames": n_frames,
            # Absolute paths work with FrameDataset's Path-based lookup.
            "path": str(video_dir.resolve()),
        })

        if i % 500 == 0:
            print(
                f"  Processed {i}/{len(video_dirs)} {class_name} videos",
                flush=True
            )

fieldnames = [
    "dataset", "method", "video_id", "label",
    "forgery_type", "split", "n_frames", "path"
]

with OUTPUT_MANIFEST.open("w", newline="") as f:
    writer = csv.DictWriter(f, fieldnames=fieldnames)
    writer.writeheader()
    writer.writerows(rows)

real_count = sum(row["label"] == 0 for row in rows)
fake_count = sum(row["label"] == 1 for row in rows)
total_frames = sum(row["n_frames"] for row in rows)

print("\n=== Manifest generation complete ===", flush=True)
print("Manifest:", OUTPUT_MANIFEST)
print("Videos:", len(rows))
print("Real videos:", real_count)
print("Fake videos:", fake_count)
print("Crops counted:", total_frames)
print("Video folders with no JPG crops:", len(skipped_videos))
print(f"Elapsed time: {time.time() - start:.1f} seconds")

if not rows or real_count == 0 or fake_count == 0:
    raise RuntimeError("Manifest is empty or one of the classes is missing.")

print("\nFirst 5 rows:")
for row in rows[:5]:
    print(row)

DFDC directory: /kaggle/input/datasets/saisdsaiiitbh/dfdc-crops/dfdc

Scanning REAL: 76 video folders

Scanning FAKE: 316 video folders

=== Manifest generation complete ===
Manifest: /kaggle/working/dfdc_manifest.csv
Videos: 392
Real videos: 76
Fake videos: 316
Crops counted: 12544
Video folders with no JPG crops: 0
Elapsed time: 3.6 seconds

First 5 rows:
{'dataset': 'dfdc', 'method': 'dfdc', 'video_id': 'abarnvbtwb', 'label': 0, 'forgery_type': 0, 'split': 'test', 'n_frames': 32, 'path': '/kaggle/input/datasets/saisdsaiiitbh/dfdc-crops/dfdc/REAL/abarnvbtwb'}
{'dataset': 'dfdc', 'method': 'dfdc', 'video_id': 'aelfnikyqj', 'label': 0, 'forgery_type': 0, 'split': 'test', 'n_frames': 32, 'path': '/kaggle/input/datasets/saisdsaiiitbh/dfdc-crops/dfdc/REAL/aelfnikyqj'}
{'dataset': 'dfdc', 'method': 'dfdc', 'video_id': 'afoovlsmtx', 'label': 0, 'forgery_type': 0, 'split': 'test', 'n_frames': 32, 'path': '/kaggle/input/datasets/saisdsaiiitbh/dfdc-crops/dfdc/REAL/afoovlsmtx'}
{'dataset': 'dfd

In [5]:
# Validate the manifest

import pandas as pd
from pathlib import Path

manifest = pd.read_csv("/kaggle/working/dfdc_manifest.csv")

required_columns = {
    "dataset", "method", "video_id", "label",
    "forgery_type", "split", "n_frames", "path"
}

assert required_columns.issubset(manifest.columns), (
    f"Missing columns: {required_columns - set(manifest.columns)}"
)

assert manifest["video_id"].is_unique, "Duplicate video IDs detected."
assert set(manifest["label"].unique()) == {0, 1}, (
    "Expected both real (0) and fake (1) labels."
)

missing_dirs = [
    p for p in manifest["path"]
    if not Path(p).is_dir()
]

assert not missing_dirs, (
    f"{len(missing_dirs)} manifest paths do not exist. "
    f"Examples: {missing_dirs[:5]}"
)

print("Manifest validation passed.")
print("\nVideo counts by label:")
print(manifest["label"].value_counts().sort_index().rename(index={0: "REAL", 1: "FAKE"}))

print("\nCrop count summary:")
print(manifest["n_frames"].describe())

print("\nSample:")
display(manifest.head())

Manifest validation passed.

Video counts by label:
label
REAL     76
FAKE    316
Name: count, dtype: int64

Crop count summary:
count    392.0
mean      32.0
std        0.0
min       32.0
25%       32.0
50%       32.0
75%       32.0
max       32.0
Name: n_frames, dtype: float64

Sample:


,dataset,method,video_id,label,forgery_type,split,n_frames,path
0,dfdc,dfdc,abarnvbtwb,0,0,test,32,/kaggle/input/datasets/saisdsaiiitbh/dfdc-crop...
1,dfdc,dfdc,aelfnikyqj,0,0,test,32,/kaggle/input/datasets/saisdsaiiitbh/dfdc-crop...
2,dfdc,dfdc,afoovlsmtx,0,0,test,32,/kaggle/input/datasets/saisdsaiiitbh/dfdc-crop...
3,dfdc,dfdc,agrmhtjdlk,0,0,test,32,/kaggle/input/datasets/saisdsaiiitbh/dfdc-crop...
4,dfdc,dfdc,ahqqqilsxt,0,0,test,32,/kaggle/input/datasets/saisdsaiiitbh/dfdc-crop...


In [10]:
from pathlib import Path

roots = [
    Path("/kaggle/input/datasets/saisdsaiiitbh/backup-1"),
    Path("/kaggle/input/datasets/saisdsaiiitbh/dinov2-base"),
    Path("/kaggle/working"),
]

for root in roots:
    if not root.exists():
        continue

    print(f"\n=== {root} ===")

    for p in root.rglob("*"):
        if p.is_file() and (
            p.name == "best.pt"
            or p.name.endswith(".pt")
            or p.name == "evaluate_dff.py"
        ):
            print(p)


=== /kaggle/input/datasets/saisdsaiiitbh/backup-1 ===
/kaggle/input/datasets/saisdsaiiitbh/backup-1/dff_full_extended/best.pt
/kaggle/input/datasets/saisdsaiiitbh/backup-1/dff_full_extended/last.pt
/kaggle/input/datasets/saisdsaiiitbh/backup-1/dff_resume_test/last.pt

=== /kaggle/input/datasets/saisdsaiiitbh/dinov2-base ===
/kaggle/input/datasets/saisdsaiiitbh/dinov2-base/dff_base_current/best.pt
/kaggle/input/datasets/saisdsaiiitbh/dinov2-base/dff_base_current/last.pt

=== /kaggle/working ===


In [11]:
# Evalutate on DinoV2-small

!python /kaggle/input/datasets/saisdsaiiitbh/dff-code/evaluate_dff.py \
    --ckpt /kaggle/input/datasets/saisdsaiiitbh/backup-1/dff_full_extended/best.pt \
    --backbone small \
    --variant dff \
    --sets /kaggle/working/dfdc_manifest.csv:all \
    --frames 0

config.json: 100%|█████████████████████████████| 547/547 [00:00<00:00, 2.44MB/s]

model.safetensors: downloading bytes: ██████████████████▋   | 74.9MB, 5.46MB/s  
model.safetensors: downloading bytes: ██████████████████████| 83.7MB, 8.02MB/s  
model.safetensors: reconstructing file: 100%|██████| 88.2MB / 88.2MB, 8.59MB/s  
Loading weights: 100%|██████████████████████| 223/223 [00:00<00:00, 2741.66it/s]
/kaggle/working/dfdc_manifest.csv:all    AUC=88.03  EER=19.74  frameAUC=79.99  (392 videos, 12544 frames)


In [12]:
# Evaluate on DinoV2-base

!python /kaggle/input/datasets/saisdsaiiitbh/dff-code/evaluate_dff.py \
    --ckpt /kaggle/input/datasets/saisdsaiiitbh/dinov2-base/dff_base_current/best.pt \
    --backbone base \
    --variant dff \
    --sets /kaggle/working/dfdc_manifest.csv:all \
    --frames 0

config.json: 100%|█████████████████████████████| 548/548 [00:00<00:00, 2.46MB/s]

model.safetensors: downloading bytes: █████████▊            |  155MB, 11.5MB/s  
model.safetensors: downloading bytes: ██████████████████    |  285MB, 23.7MB/s  
model.safetensors: downloading bytes: ████████████████████  |  315MB, 26.7MB/s  
model.safetensors: downloading bytes: ██████████████████████|  328MB, 30.3MB/s  
model.safetensors: reconstructing file: 100%|██████|  346MB /  346MB, 32.7MB/s  
Loading weights: 100%|██████████████████████| 223/223 [00:00<00:00, 2449.49it/s]
/kaggle/working/dfdc_manifest.csv:all    AUC=84.00  EER=23.68  frameAUC=78.10  (392 videos, 12544 frames)
